# fase_3 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Migrasi Rekrutmen & Pelamar dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector 
import pandas as pd
import re
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_future'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_future']['database']}")

Connected to dataleap_v5_example_new and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('pengajuan', 'pengajuan_karyawan'),
    ('histori_pengajuan', 'histori_pengajuan'),
    ('pelamar', 'pelamar'),
    ('pekerjaan', 'pelamar_kerja'),
    ('pendidikan', 'pelamar_sekolah'),
    ('kursus', 'pelamar_kursus'),
    ('pelamar_note', 'progres_pelamar'),
    ('pelamar_users', 'rekrutmen_pelamar')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    cursor_old.execute(f"SELECT * FROM `{old_t}`")
    raw_data[old_t] = cursor_old.fetchall()
    print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")

✅ pengajuan loaded: 33 records
✅ histori_pengajuan loaded: 79 records
✅ pelamar loaded: 178 records
✅ pekerjaan loaded: 67 records
✅ pendidikan loaded: 53 records
✅ kursus loaded: 50 records
✅ pelamar_note loaded: 403 records
✅ pelamar_users loaded: 281 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# --- HELPER FUNCTIONS ---
# ponytail: gender manual lookup table for NULLs
GENDER_MAP = {
    'ditari@leapsurabaya.sch.id': 'Perempuan',
    'safitriintan801@gmail.com': 'Perempuan',
    'ddoanda@gmail.com': 'Laki laki',
    'raputri.rap@gmail.com': 'Perempuan',
    'gistara.azzahra@gmail.com': 'Perempuan',
    'tedialvianto062@gmail.com': 'Laki laki',
    'nimasbuwana@gmail.com': 'Perempuan',
    'erfiadyntahzanin@gmail.com': 'Perempuan',
    'bryantfrederico@gmail.com': 'Laki laki',
    'ficcaayu@gmail.com': 'Perempuan',
    'graciela@leapsurabaya.sch.id': 'Perempuan',
    '09putrirahayu@gmail.com': 'Perempuan',
    'shaniafebrianaa@gmail.com': 'Perempuan',
    'bibah@gmail.com': 'Perempuan',
    'admin@gmail.com': 'Perempuan',
    'mochamadsaifulr15@gmail.com': 'Laki laki',
    'akin@email.com': 'Laki laki',
    'staffhrd@leapsurabaya.sch.id': 'Laki laki',
    'rini.rahayu@leapsurabaya.sch.id': 'Perempuan',
    'cantikaswasti76@gmail.com': 'Perempuan',
    'hartatik@leapsurabaya.sch.id': 'Perempuan',
    'nisrina.dea@leapsurabaya.sch.id': 'Perempuan',
    'miekepuspita@leapsurabaya.sch.id': 'Perempuan',
    'agung.wijayanto@leapsurabaya.sch.id': 'Laki laki',
    'qorin.rahmaniah@leapsurabaya.sch.id': 'Perempuan',
    'miftakhul.jannah@leapsurabaya.sch.id': 'Perempuan',
    'vivi.wulandari@leapsurabaya.sch.id': 'Perempuan',
    'eka.wahyuni@leapsurabaya.sch.id': 'Perempuan',
    'siti.uswatun@leapsurabaya.sch.id': 'Perempuan',
    'ericasusanto@leapsurabaya.sch.id': 'Perempuan',
    'getari@leapsurabaya.sch.id': 'Perempuan',
    'generalaffair@gmail.com': 'Laki laki'
}

def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\\d+', str(s))
    return int(nums[0]) if nums else None

def extract_place(ttl):
    if pd.isna(ttl) or not str(ttl).strip(): return None
    s = str(ttl).strip()
    if ',' in s:
        return s.split(',')[0].strip()
    match = re.search(r'^[^0-9]+', s)
    return match.group(0).strip() if match else s

def extract_date(ttl):
    if pd.isna(ttl) or not str(ttl).strip(): return None
    s = str(ttl).strip()
    if ',' in s:
        parts = s.split(',')
        if len(parts) > 1: return parts[1].strip()
    match = re.search(r'(\\d.*)', s)
    return match.group(1).strip() if match else None

def parse_date(date_str):
    if pd.isna(date_str): return None
    s = str(date_str).strip()
    if s in ('', '-', '0', 'nan', 'NaN'): return None
    
    # First normalize day-of-month ranges: "27 - 29 Mei 2023" -> "27 Mei 2023"
    s = re.sub(r'\\b(\\d{1,2})\\s*-\\s*\\d{1,2}\\b', r'\\1', s)
    
    # Try to see if it's a month-level range: e.g. "29 September - 6 Oktober 2021"
    parts = re.split(r'\\s+-\\s+|\\s+(?:sd|s/d|dan|s\\.d\\.)\\s+', s, flags=re.IGNORECASE)
    if len(parts) > 1:
        part1 = parts[0].strip()
        part2 = parts[1].strip()
        if not re.search(r'\\b\\d{4}\\b', part1):
            year_match = re.search(r'\\b\\d{4}\\b', part2)
            if year_match:
                part1 = part1 + " " + year_match.group(0)
        s = part1
        
    months_id_to_en = {
        'januari': 'January', 'februari': 'February', 'maret': 'March', 'april': 'April',
        'mei': 'May', 'juni': 'June', 'juli': 'July', 'agustus': 'August',
        'september': 'September', 'oktober': 'October', 'november': 'November', 'desember': 'December',
        'jan': 'January', 'feb': 'February', 'mar': 'March', 'apr': 'April', 'jun': 'June',
        'jul': 'July', 'agu': 'August', 'agst': 'August', 'sep': 'September', 'okt': 'October', 'nov': 'November', 'des': 'December'
    }
    
    s = s.replace('-', ' ').replace('/', ' ').replace('.', '').strip()
    s = re.sub(r'\\s+', ' ', s)
    
    for id_m, en_m in months_id_to_en.items():
        s = re.sub(rf'\\b{id_m}\\b', en_m, s, flags=re.IGNORECASE)
        
    year_match = re.search(r'\\b\\d{4}\\b', s)
    if not year_match:
        s = s + " 2023"
        
    formats = [
        '%d %B %Y', '%d %b %Y', '%B %Y', '%b %Y',
        '%d %m %Y', '%m %d %Y', '%Y %m %d',
        '%d %m %y', '%m %d %y', '%y %m %d'
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(s, format=fmt).date()
        except:
            continue
            
    try:
        res = pd.to_datetime(s, errors='coerce')
        if pd.notna(res):
            return res.date()
    except:
        pass
        
    return None

def extract_latest_year(tahun_str):
    if pd.isna(tahun_str) or not str(tahun_str).strip(): return None
    years = re.findall(r'\\d{4}', str(tahun_str))
    if years:
        return max(map(int, years))
    return None

def clean_currency(val):
    if pd.isna(val): return 0
    s = str(val).strip()
    nums = re.sub(r'[^0-9]', '', s)
    return int(nums) if nums else 0

def clean_ipk(val):
    if pd.isna(val): return 0.0
    s = str(val).strip().replace(',', '.')
    match = re.search(r'\\d+\\.?\\d*', s)
    if match:
        try: return float(match.group(0))
        except: return 0.0
    return 0.0

def clean_name_without_titles(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    if ',' in s: 
        s = s.split(',')[0]
    titles = [
        r'\\bs\\.?\\s*pd\\b', r'\\bm\\.?\\s*pd\\b', r'\\bs\\.?\\s*s\\b', r'\\bs\\.?\\s*t\\b', 
        r'\\bs\\.?\\s*hum\\b', r'\\bs\\.?\\s*kom\\b', r'\\ba\\.?\\s*md\\b', r'\\bs\\.?\\s*e\\b', 
        r'\\bm\\.?\\s*m\\b', r'\\bdr\\b', r'\\bdra\\b', r'\\bdrs\\b', r'\\bprof\\b',
        r'\\bpsi\\b', r'\\bs\\.?\\s*psi\\b'
    ]
    for title in titles:
        s = re.sub(title, '', s, flags=re.IGNORECASE)
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

# --- TRANSFORMATION ---

# 1. pengajuan -> pengajuan_karyawan
if 'pengajuan' in raw_data:
    df = pd.DataFrame(raw_data['pengajuan'])
    df['status'] = df['status'].replace('Sudah Direvisi', 'Sudah Revisi')
    df['id_pengajuan_new'] = df.index + 1
    pengajuan_id_map = dict(zip(df['idpengajuan'], df['id_pengajuan_new']))
    
    # ponytail: build and save pengajuan_karyawan ID mapping (old ID -> new auto-incremented ID)
    df_mapping_peng = pd.DataFrame({
        'idpengajuan_lama': df['idpengajuan'],
        'id_pengajuan_baru': df['id_pengajuan_new']
    })
    df_mapping_peng['id_pengajuan_baru'] = df_mapping_peng['id_pengajuan_baru'].astype('Int64')
    pd.to_pickle(df_mapping_peng, 'mapping_pengajuan_karyawan.pkl')
    transformed_dfs['mapping_pengajuan_karyawan'] = df_mapping_peng
    
    df['id_user'] = df['idusers']
    mapping = {
        'id_user': 'id_user', 'keterangan': 'posisi',
        'jumlah': 'jumlah', 'syarat': 'syarat', 'pertanyaan': 'pertanyaan',
        'alur': 'alur_seleksi', 'test': 'daftar_tes', 'status': 'status',
        'created_at': 'created_at'
    }
    transformed_dfs['pengajuan_karyawan'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 2. histori_pengajuan -> histori_pengajuan
if 'histori_pengajuan' in raw_data:
    df = pd.DataFrame(raw_data['histori_pengajuan'])
    df['status'] = df['status'].replace('Sudah Direvisi', 'Sudah Revisi')
    df['id_pengajuan'] = df['idpengajuan'].map(pengajuan_id_map).astype('Int64')
    mapping = {
        'id_pengajuan': 'id_pengajuan',
        'status': 'status_verifikasi_pengajuan', 'catatan': 'catatan',
        'created_at': 'created_at'
    }
    transformed_dfs['histori_pengajuan'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 3. pelamar -> pelamar
if 'pelamar' in raw_data:
    df_pel = pd.DataFrame(raw_data['pelamar'])
    df_pel['tempat_lahir'] = df_pel['ttl'].apply(extract_place)
    df_pel['tanggal_lahir'] = df_pel['ttl'].apply(extract_date).apply(parse_date)
    
    def map_nikah(x):
        val = str(x).strip().lower()
        if val in ['menikah', 'nikah', 'kawin']: return 'Menikah'
        return 'Belum Menikah'
    
    df_pel['status_pernikahan'] = df_pel['statusnikah'].apply(map_nikah)
    df_pel['penggunaan_laptop'] = df_pel['gunalaptop'].apply(lambda x: 'Pernah' if str(x).strip().lower() in ['pernah', 'ya, pernah', 'ya'] else 'Tidak Pernah')
    df_pel['gaji'] = df_pel['gaji'].apply(clean_currency)
    
    # ponytail: clean and normalize gender
    def clean_gender(row):
        email = str(row.get('email', '')).strip().lower()
        jk = row.get('jk')
        if pd.notna(jk) and str(jk).strip():
            val = str(jk).strip().lower()
            if 'perempuan' in val or val == 'p': return 'Perempuan'
            if 'laki' in val or val == 'l': return 'Laki laki'
        return GENDER_MAP.get(email, 'Laki laki')
    df_pel['jenis_kelamin'] = df_pel.apply(clean_gender, axis=1)
    
    cursor_old.execute("SELECT idusers, email, nama FROM users")
    df_users = pd.DataFrame(cursor_old.fetchall())
    cursor_old.execute("SELECT idpelamar, idusers FROM pelamar_users")
    df_pu = pd.DataFrame(cursor_old.fetchall())
    
    cursor_old.execute("SELECT DISTINCT idusers FROM pekerjaan")
    df_pekerjaan_users = pd.DataFrame(cursor_old.fetchall())
    cursor_old.execute("SELECT DISTINCT idusers FROM pendidikan")
    df_pendidikan_users = pd.DataFrame(cursor_old.fetchall())
    cursor_old.execute("SELECT DISTINCT idusers FROM kursus")
    df_kursus_users = pd.DataFrame(cursor_old.fetchall())
    
    child_users = set(df_pekerjaan_users['idusers']).union(
        set(df_pendidikan_users['idusers'])
    ).union(
        set(df_kursus_users['idusers'])
    )
    
    def clean_str(s):
        if pd.isna(s): return ""
        return str(s).strip().lower()
        
    df_users['email_clean'] = df_users['email'].apply(clean_str)
    df_users['name_clean'] = df_users['nama'].apply(clean_name_without_titles)
    
    df_pel['email_clean'] = df_pel['email'].apply(clean_str)
    df_pel['name_clean'] = df_pel['nama'].apply(clean_name_without_titles)
    
    pu_map = dict(zip(df_pu['idusers'], df_pu['idpelamar']))
    email_to_pelamar = {}
    for _, row in df_pel.iterrows():
        email = row['email_clean']
        if email and email not in email_to_pelamar:
            email_to_pelamar[email] = row['idpelamar']
            
    name_to_pelamar = {}
    for _, row in df_pel.iterrows():
        name = row['name_clean']
        if name and name not in name_to_pelamar:
            name_to_pelamar[name] = row['idpelamar']
            
    user_to_pelamar_id = {}
    unmatched_users = []
    
    for u_id in child_users:
        u_rows = df_users[df_users['idusers'] == u_id]
        if u_rows.empty:
            unmatched_users.append((u_id, "User not in users table", ""))
            continue
        u_row = u_rows.iloc[0]
        u_email = u_row['email_clean']
        u_name = u_row['name_clean']
        
        p_id = pu_map.get(u_id)
        if p_id:
            user_to_pelamar_id[u_id] = p_id
            continue
            
        p_id = email_to_pelamar.get(u_email)
        if p_id:
            user_to_pelamar_id[u_id] = p_id
            continue
            
        p_id = name_to_pelamar.get(u_name)
        if p_id:
            user_to_pelamar_id[u_id] = p_id
            continue
            
        unmatched_users.append((u_id, u_row['nama'], u_row['email']))
        
    df_pel_extended = df_pel.copy()
    for u_id, name, email in unmatched_users:
        # ponytail: fill missing user info
        cleaned_name = name if pd.notna(name) else '-'
        new_row = {
            'idpelamar': u_id,
            'nama': cleaned_name,
            'email': email,
            'idpengajuan': None,
            'jenis_kelamin': GENDER_MAP.get(email, 'Laki laki')
        }
        df_pel_extended = pd.concat([df_pel_extended, pd.DataFrame([new_row])], ignore_index=True)
        
    df_pel_extended['id_pelamar_new'] = df_pel_extended.index + 1
    pelamar_id_map = dict(zip(df_pel_extended['idpelamar'], df_pel_extended['id_pelamar_new']))
    
    # ponytail: build and save pelamar ID mapping (old ID -> new auto-incremented ID)
    df_mapping_pel = pd.DataFrame({
        'idpelamar_lama': df_pel_extended['idpelamar'],
        'id_pelamar_baru': df_pel_extended['id_pelamar_new']
    })
    df_mapping_pel['id_pelamar_baru'] = df_mapping_pel['id_pelamar_baru'].astype('Int64')
    pd.to_pickle(df_mapping_pel, 'mapping_pelamar.pkl')
    transformed_dfs['mapping_pelamar'] = df_mapping_pel
    
    final_user_to_pelamar_id = {}
    for u_id in child_users:
        old_p_id = user_to_pelamar_id.get(u_id)
        if old_p_id:
            final_user_to_pelamar_id[u_id] = pelamar_id_map.get(old_p_id)
        else:
            final_user_to_pelamar_id[u_id] = pelamar_id_map.get(u_id)
            
    df_pel_extended['id_pelamar'] = df_pel_extended['id_pelamar_new']
    df_pel_extended['id_pengajuan'] = df_pel_extended['idpengajuan'].map(pengajuan_id_map).astype('Int64')
    
    # ponytail: fill missing NOT NULL columns in pelamar
    df_pel_extended['email'] = df_pel_extended['email'].fillna('-')
    df_pel_extended['nama'] = df_pel_extended['nama'].fillna('-')
    df_pel_extended['panggilan'] = df_pel_extended['panggilan'].fillna('-')
    df_pel_extended['tempat_lahir'] = df_pel_extended['tempat_lahir'].fillna('-')
    df_pel_extended['tanggal_lahir'] = df_pel_extended['tanggal_lahir'].fillna(pd.to_datetime('1970-01-01').date())
    df_pel_extended['status_pernikahan'] = df_pel_extended['status_pernikahan'].fillna('Belum Menikah')
    df_pel_extended['penggunaan_laptop'] = df_pel_extended['penggunaan_laptop'].fillna('Tidak Pernah')
    df_pel_extended['gaji'] = df_pel_extended['gaji'].fillna(0)
    
    for text_col in ['alamat', 'domisili', 'wa', 'ig', 'fb', 'sosmed', 'laptop', 'internet', 'kegiatan', 'rencana', 'mobilitas', 'info', 'wfo', 'jenis', 'work', 'ppdk', 'pengalaman', 'wawasan', 'sehat', 'ajar', 'app', 'apps', 'link', 'resign', 'piciq', 'picminat', 'picpribadi']:
        df_pel_extended[text_col] = df_pel_extended[text_col].fillna('-')
        
    # ponytail: apply custom shortened WFO responses to fit within 50 chars limit without losing context
    WFO_CLEAN_MAP = {
        "Ya, karena jarak antara kantor LEAP dengan tempat kerja saya tidak terlalu jauh sehingga memungkinkan untuk saya bekerja secara offline. Selain itu, LEAP memberikan fleksibilitas dalam mengatur jam mengajar yang dibagi ke dalam 3 sesi sehingga saya bisa memilih sesi mana yang sesuai dengan jadwal harian saya. Selain itu, LEAP memberikan jadwal bekerja dari hari Senin hingga Jumat saja yang berarti institusi ini menghargai jasa seorang guru yang seharusnya bekerja di hari efektif dan bukan di akhir pekan.": "Ya, lokasi dekat & jadwal fleksibel",
        "Ya, bekerja secara offline di kantor LEAP memberikan banyak manfaat bagi saya dan LEAP. Saya berkomitmen untuk memberikan kontribusi terbaik dan membantu LEAP mencapai tujuannya. Karena dapat berinteraksi langsung dengan tim LEAP terutama dalam meningkatkan kolaborasi dan mempermudah penyelesaian masalah. Saya dapat memberikan layanan yang lebih personal dan efektif kepada siswa serta memahami kebutuhan dan kendala. Bekerja di kantor menunjukkan komitmen saya terhadap LEAP dan pekerjaannya. ": "Ya, siap kolaborasi offline di kantor",
        "Ya, saya berdomisili Surabaya, area Mojo. Saya memiliki motor pribadi, dan untuk menuju kantor LEAP membutuhkan waktu cepat, sekitar 20 menit. Dengan sks lulus 121/144, beban kuliah saya ringan untuk semester ini. Karena hal-hal diatas, saya akan senang hati untuk bisa bekerja sama dan bertemu rekan-rekan di LEAP": "Ya, domisili Mojo, motor pribadi, kuliah ringan",
        "Ya, saya bersedia untuk bekerja secara offline (WFO) di kantor LEAP Surabaya. Saya telah mempertimbangkan kebutuhan ini sejak awal dan siap untuk menetap di Surabaya jika diterima, agar dapat menjalankan peran dan tanggung jawab secara maksimal sesuai dengan budaya kerja dan kebutuhan tim LEAP.": "Ya, bersedia WFO & siap menetap di Surabaya",
        "Ya, saya bersedia bekerja secara offline (WFO) di kantor LEAP di Rungkut Asri Tengah VII/51, Surabaya. Saya memahami bahwa bekerja langsung di kantor bisa membantu saya lebih banyak belajar, berkoordinasi lebih baik, dan mendapatkan pengalaman nyata di dunia kerja": "Ya, bersedia WFO Rungkut untuk cari pengalaman",
        "Tidak. Karena saya single mom disini dan ada dua anak dibawah umur tanpa asisten. Sementara suami saya bekerja di Jakarta dan pulang sebulan sekali. Saya diamanahi untuk stay di rumah saja meski bekerja.": "Tidak, asuh 2 anak di rumah & suami di Jakarta",
        "Ya, saya bersedia. Saya berdomisili di Surabaya dan memiliki akses transportasi yang memadai, sehingga dapat memastikan kehadiran tepat waktu di kantor sesuai jadwal kerja yang ditetapkan.": "Ya, domisili Surabaya & transport memadai",
        "Ya, saya bersedia untuk bekerja secara offline (WFO) di lokasi kantor LEAP di Rungkut Asri Tengah VII/51, Surabaya, karena lokasinya cukup dekat dengan akomodasi saya.": "Ya, bersedia WFO, dekat dengan akomodasi",
        "Ya saya bersedia untuk bekerja secara offline di surabaya, karena dengan bekerja di surabaya akses untuk transportasi lebih mudah serta lebih dekat dengan rumah.": "Ya, WFO Surabaya, transport mudah & dekat rumah",
        "Ya, saya bersedia. Karena menurut saya, bekerja secara offline lebih menyenangkan karena sambil mendapat ilmu juga bisa berinteraksi dengan relasi lainnya.": "Ya, interaksi offline menyenangkan & dapat ilmu",
        "Ya, karena saya bisa mengajar secara offline (WFO) di lokasi tersebut, dan ingin menambah pengalaman bekerja dan mengajar Bahasa Inggris di sebuah lembaga.": "Ya, ingin tambah pengalaman mengajar offline",
        "Ya, saat ini saya tengah difasilitasi oleh Rumah Gemilang Indonesia di daerah Kenjeran, Surabaya dan jaraknya relatif dekat dengan domisili saya sekarang.": "Ya, tinggal di RGI Kenjeran, dekat lokasi",
        "Ya, saya bersedia karena saya merasa jika bekerja offline dapat membuat saya lebih fokus, produktif, dan mudah berkoordinasi secara langsung dengan tim.": "Ya, WFO lebih fokus, produktif & koordinasi",
        "Ya bersedia, sehubungan dengan jarak yang masih bisa dijangkau, bekerja secara offline tidak menjadi hambatan saya untuk meninggalkan pekerjaan ini": "Ya, jarak terjangkau & bukan hambatan",
        "Ya, saya bersedia bekerja secara offline di kantor LEAP untuk mendapatkan pengalaman kerja langsung dan berinteraksi dengan tim": "Ya, cari pengalaman langsung & interaksi tim",
        "tidak, karena setelah saya tanya dengan kakak admin mengenai posisi yang saya lamar sepertinya tidak membutuhkan untuk wfo": "Tidak, kata admin posisi tidak butuh WFO",
        "Ya. kalau offline bersedia WFO. kalau online mungkin bisa lebih fleksibel karena saya masih harus menyelesaikan thesis.": "Ya WFO, tapi fleksibel sambil selesai thesis",
        "Ya, karena saya disini sedang merantau dan saya tidak mempunyai laptop atau komputer jadi saya tidak bisa kerja remote": "Ya, sedang merantau & tidak ada laptop remote",
        "saya bersedia selama masih dalam jangkauan kemampuan saya dan tidak mengganggu jam mengajar saya di sekolah": "Ya, asal tidak ganggu jam mengajar sekolah",
        "Ya, dikarenakan sesuai dengan passion dan pengalaman kerja saya, serta jarak dari rumah saya sangat dekat. ": "Ya, sesuai passion & jarak dekat dari rumah",
        "Ya, Saya bersedia bekerja secara offline (WFO) di lokasi kantor LEAP (Rungkut Asri Tengah VII/51, Surabaya)": "Ya, bersedia WFO Rungkut Asri Surabaya",
        "Ya, saya memiliki waktu dan kemampuan untuk bisa hadir secara offline di kantor maksimal sampai jam 4 sore": "Ya, bisa hadir offline maksimal jam 4 sore",
        "Iya bersedia WFO (disesuaikan dengan jadwal pekerjaan utama saya) dan ketentuan yang telah disepakati.": "Ya, sesuaikan jadwal pekerjaan utama",
        "Tidak, untuk saat ini saya lebih memilih WFA dikarenakan saya dalam kondisi sedang merawat orang tua.": "Tidak, pilih WFA karena sedang rawat ortu",
        "Ya, karena saya disini merantau dan tidak punya laptop atau komputer jadi tidak dapat bekerja remote": "Ya, merantau & tidak punya laptop remote",
        "Ya, jikalau jadwal bekerja tidak bertabrakan dengan jadwal kuliah maka saya siap bekerja secara WFO": "Ya, jika tidak tabrakan dengan kuliah",
        "Ya,saya yakin bisa membantu meningkatkan skill saya kedepannya jika kegiatan magangnya offline": "Ya, magang offline tingkatkan skill diri",
        "Ya, karena saya sudah terbiasa dengan lingkup kerja yang ramai and mengutamakan kerja sama tim": "Ya, terbiasa kerja tim & lingkup ramai",
        "Ya, karena saya sudah terbiasa dengan lingkup kerja yang ramai dan mengutamakan kerja sama tim": "Ya, terbiasa kerja tim & lingkup ramai",
        "Ya, Karena saya dapat bekerja sama dengan tim dan terbiasa bekerja fulltime secara offline ": "Ya, biasa fulltime offline & kerja tim",
        "Ya, jika jadwal tersebut tidak bertabrakan dengan jadwal kuliah saya maka saya bersedia WFO": "Ya, asal tidak tabrakan dengan kuliah",
        "Ya, karena saya saat ini bedomisili di Surabaya yang mempermudah akses saya ke kantor": "Ya, domisili Surabaya mudahkan akses kantor",
        "Ya. Saya tidak keberatan dengan lokasi dan sistem kerja offline seperti demikian.": "Ya, tidak keberatan sistem kerja offline",
        "ya, saya bersedia bekerja offline menyelesaikan waktu kuliah saya (parttime)": "Ya, bersedia parttime offline sambil kuliah",
        "Ya, saya bersedia untuk bekerja secara offline (WFO) di lokasi kantor LEAP": "Ya, bersedia WFO di lokasi kantor LEAP",
        "Ya, karena saya ingin mempunyai pengalaman secara langsung dalam bekerja.": "Ya, ingin pengalaman kerja langsung",
        "Ya, karena saya berdomisili di Surabaya dan memiliki kendaraan pribadi.": "Ya, domisili Surabaya & motor pribadi",
        "iya, saya siap untuk bekerja secara offline di kantor LEAP Surabaya": "Ya, siap offline di kantor LEAP Surabaya",
        "ya bersedia, bekerja WFO di LEAP cabang Pesanggaran, Banyuwangi": "Ya, bersedia WFO di LEAP Banyuwangi",
        "Ya,untuk bekerja saya tidak masalah baik offline maupun online": "Ya, tidak masalah offline maupun online",
        "Ya karena saya juga ingin bisa berinteraksi dengan orang lain": "Ya, ingin berinteraksi dengan orang lain",
        "Ya, karena saya ingin berinteraksi langsung dengan anak-anak": "Ya, ingin berinteraksi dengan anak-anak",
        "Tidak, untuk freelance saya terbiasa handle secara remote": "Tidak, freelance biasa remote saja",
        "bersedia apabila jam kerja di luar jam kerja utama saya.": "Ya, asal di luar jam kerja utama saya",
        "Tidak karena posisi yang ditawarkan bersifat freelance": "Tidak, karena posisi freelance",
        "Ya, Ingin mempunyai pengalaman langsung di dunia kerja": "Ya, ingin pengalaman langsung dunia kerja",
        "Iya, karena rumah saya hanya 10 menit dari kantor LEAP": "Ya, rumah dekat hanya 10 menit dari kantor",
        "Ya, jika saya keterima saya akan menetap di surabaya": "Ya, siap menetap di Surabaya jika diterima",
        "Ya untuk memulai hari pekerjaan magang di luar rumah": "Ya, untuk mulai magang di luar rumah",
        "Tidak, karena penempatan di Banyuwangi (Pesanggaran)": "Tidak, karena penempatan di Banyuwangi",
        "Ya, saya bisa mengoptimalkan bekerja darimana saja.": "Ya, bisa optimalkan kerja dari mana saja"
    }
    df_pel_extended['wfo'] = df_pel_extended['wfo'].astype(str).str.strip().replace(WFO_CLEAN_MAP)
        
    # ponytail: safely truncate string columns to fit database varchar limits
    for col_name, limit in [
        ('email', 150), ('nama', 150), ('panggilan', 50), ('tempat_lahir', 100),
        ('wa', 20), ('linkedin', 255), ('ig', 100), ('fb', 100), ('internet', 100),
        ('info', 100), ('wfo', 50), ('jenis', 50), ('link', 255),
        ('piciq', 255), ('picminat', 255), ('picpribadi', 255)
    ]:
        if col_name in df_pel_extended.columns:
            df_pel_extended[col_name] = df_pel_extended[col_name].apply(lambda x: str(x)[:limit] if pd.notna(x) else x)
            
    # ponytail: clean and convert integer columns to prevent string values like 'asd'
    df_pel_extended['toefl'] = pd.to_numeric(df_pel_extended['toefl'], errors='coerce').fillna(0).astype(int)
    df_pel_extended['hasiliq'] = pd.to_numeric(df_pel_extended['hasiliq'], errors='coerce').fillna(0).astype(int)
    df_pel_extended['bergabung'] = df_pel_extended['bergabung'].fillna(pd.to_datetime('1970-01-01').date())
    # ponytail: use safe modern date for created_at to avoid MySQL TIMESTAMP out of range errors due to timezone conversion
    df_pel_extended['created_at'] = df_pel_extended['created_at'].fillna(pd.to_datetime('2020-01-01 00:00:00'))
    
    mapping = {
        'id_pengajuan': 'id_pengajuan', 'email': 'email_pelamar',
        'nama': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jenis_kelamin': 'jenis_kelamin',
        'tempat_lahir': 'tempat_lahir', 'tanggal_lahir': 'tanggal_lahir',
        'alamat': 'alamat_ktp', 'domisili': 'alamat_domisili', 'wa': 'nomor_wa',
        'linkedin': 'akun_linkedin', 'ig': 'akun_instagram', 'fb': 'akun_facebook', 
        'sosmed': 'sosmed_lain', 'laptop': 'spesifikasi_laptop', 'internet': 'internet',
        'kegiatan': 'kegiatan_sekarang', 'rencana': 'rencana_karir', 'mobilitas': 'mobilitas',
        'info': 'sumber_info', 'wfo': 'siap_wfo', 'bergabung': 'tanggal_bergabung',
        'jenis': 'kategori_pelamar', 'work': 'riwayat_kerja', 'ppdk': 'riwayat_pendidikan',
        'pengalaman': 'pengalaman_bidang', 'wawasan': 'wawasan', 'sehat': 'riwayat_kesehatan',
        'status_pernikahan': 'status_pernikahan', 'ajar': 'kemampuan_ajar', 'app': 'penguasaan_aplikasi', 
        'apps': 'aplikasi_lainnya', 'penggunaan_laptop': 'penggunaan_laptop', 'toefl': 'skor_toefl',
        'gaji': 'ekspektasi_gaji', 'link': 'tautan_berkas', 'resign': 'alasan_resign',
        'hasiliq': 'skor_iq', 'piciq': 'foto_iq', 'picminat': 'foto_minat', 
        'picpribadi': 'foto_kepribadian', 'created_at': 'created_at'
    }
    transformed_dfs['pelamar'] = df_pel_extended.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 4. pekerjaan -> pelamar_kerja
if 'pekerjaan' in raw_data:
    df = pd.DataFrame(raw_data['pekerjaan'])
    df['id_pelamar'] = df['idusers'].map(final_user_to_pelamar_id).astype('Int64')
    mapping = {
        'id_pelamar': 'id_pelamar',
        'namaperusahaan': 'nama_perusahaan', 'periode': 'periode', 'jabatan': 'jabatan',
        'jobdesk': 'deskripsi_kerja'
    }
    transformed_dfs['pelamar_kerja'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 5. pendidikan -> pelamar_sekolah
if 'pendidikan' in raw_data:
    df = pd.DataFrame(raw_data['pendidikan'])
    df['tahun'] = df['tahun'].apply(extract_latest_year).fillna(2000).astype(int)
    df['ipk'] = df['ipk'].apply(clean_ipk).fillna(0.0)
    df['id_pelamar'] = df['idusers'].map(final_user_to_pelamar_id).astype('Int64')
    
    for col in ['sekolah', 'jenjang', 'prodi', 'organisasi']:
        df[col] = df[col].fillna('-')
        
    mapping = {
        'id_pelamar': 'id_pelamar',
        'sekolah': 'nama_sekolah', 'jenjang': 'jenjang', 'prodi': 'prodi',
        'tahun': 'tahun_lulus', 'ipk': 'ipk', 'organisasi': 'organisasi'
    }
    transformed_dfs['pelamar_sekolah'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 6. kursus -> pelamar_kursus
if 'kursus' in raw_data:
    df = pd.DataFrame(raw_data['kursus'])
    df['tanggal'] = df['tanggal'].apply(parse_date).fillna(pd.to_datetime('1970-01-01').date())
    df['id_pelamar'] = df['idusers'].map(final_user_to_pelamar_id).astype('Int64')
    
    for col in ['nama', 'deskripsi', 'lokasi', 'nosertifikat']:
        df[col] = df[col].fillna('-')
        
    mapping = {
        'id_pelamar': 'id_pelamar',
        'nama': 'nama_kursus', 'tanggal': 'tanggal', 'deskripsi': 'deskripsi',
        'lokasi': 'lokasi', 'nosertifikat': 'nomor_sertifikat'
    }
    transformed_dfs['pelamar_kursus'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 7. pelamar_note -> progres_pelamar
if 'pelamar_note' in raw_data:
    df = pd.DataFrame(raw_data['pelamar_note'])
    df['status'] = df['status'].replace('baru', 'Baru')
    df['id_pelamar'] = df['idpelamar'].map(pelamar_id_map).astype('Int64')
    df['id_user'] = df['idusers']
    
    for col in ['note', 'link', 'pertanyaan']:
        df[col] = df[col].fillna('-')
        
    mapping = {
        'id_pelamar': 'id_pelamar',
        'id_user': 'id_user', 'status': 'status_progres_pelamar',
        'note': 'catatan', 'link': 'tautan_file', 'pertanyaan': 'pertanyaan',
        'created_at': 'created_at'
    }
    transformed_dfs['progres_pelamar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 8. pelamar_users -> rekrutmen_pelamar
if 'pelamar_users' in raw_data:
    df = pd.DataFrame(raw_data['pelamar_users'])
    df['id_pelamar'] = df['idpelamar'].map(pelamar_id_map).astype('Int64')
    df['id_user'] = df['idusers']
    mapping = {
        'id_pelamar': 'id_pelamar', 'id_user': 'id_user'
    }
    transformed_dfs['rekrutmen_pelamar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# ponytail: auto-convert datetime and date columns to standard strings to avoid MySQL timestamp conversion errors
for table_name, df_tbl in list(transformed_dfs.items()):
    if df_tbl is not None and not df_tbl.empty:
        for col in df_tbl.columns:
            if pd.api.types.is_datetime64_any_dtype(df_tbl[col]):
                df_tbl[col] = df_tbl[col].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) else None)
            else:
                first_val = df_tbl[col].dropna().iloc[0] if not df_tbl[col].dropna().empty else None
                if first_val is not None and hasattr(first_val, 'strftime'):
                    import datetime as dt_mod
                    if isinstance(first_val, dt_mod.datetime) or hasattr(first_val, 'hour'):
                        df_tbl[col] = df_tbl[col].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) and hasattr(x, 'strftime') else (str(x) if pd.notna(x) else None))
                    else:
                        df_tbl[col] = df_tbl[col].apply(lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) and hasattr(x, 'strftime') else (str(x) if pd.notna(x) else None))

print(f"OK: Transformasi {len(transformed_dfs)} tabel Fase 3 selesai.")


OK: Transformasi 10 tabel Fase 3 selesai.


## 3.1 Verifikasi Hasil Transformasi
Bagian ini menampilkan perbandingan jumlah data dan tipe data untuk pengecekan manual.

In [5]:
# 3.1.1 Ringkasan Jumlah Baris
print("📊 RINGKASAN MIGRASI (RECORDS COUNT)")
print("="*70)
summary_list = []
for old_t, new_t in hanif_tables_map:
    old_c = len(raw_data.get(old_t, []))
    new_c = len(transformed_dfs.get(new_t, []))
    summary_list.append({
        'Tabel Lama': old_t,
        'Tabel Baru': new_t,
        'Old Recs': old_c,
        'New Recs': new_c,
        'Diff': new_c - old_c,
        'Status': "✅ OK" if old_c == new_c else "⚠️ Cek"
    })
display(pd.DataFrame(summary_list))

total_old_all = sum(len(records) for records in raw_data.values())
total_new_all = sum(len(df) for df in transformed_dfs.values())
print(f"\n📢 TOTAL REKAPITULASI: {total_old_all} (Old) ➔ {total_new_all} (New)")
if total_old_all == total_new_all: print("✅ SEMUA DATA TERANGKUT")
else: print(f"⚠️ ADA SELISIH: {total_new_all - total_old_all} baris")

📊 RINGKASAN MIGRASI (RECORDS COUNT)


,Tabel Lama,Tabel Baru,Old Recs,New Recs,Diff,Status
0,pengajuan,pengajuan_karyawan,33,33,0,✅ OK
1,histori_pengajuan,histori_pengajuan,79,79,0,✅ OK
2,pelamar,pelamar,178,192,14,⚠️ Cek
3,pekerjaan,pelamar_kerja,67,67,0,✅ OK
4,pendidikan,pelamar_sekolah,53,53,0,✅ OK
5,kursus,pelamar_kursus,50,50,0,✅ OK
6,pelamar_note,progres_pelamar,403,403,0,✅ OK
7,pelamar_users,rekrutmen_pelamar,281,281,0,✅ OK



📢 TOTAL REKAPITULASI: 1144 (Old) ➔ 1383 (New)
⚠️ ADA SELISIH: 239 baris


In [6]:
# 3.1.2 Output Pengecekan Kolom Spesifik (Keterangan Mapping)
print("\n🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)")
print("="*70)

# 1. Pengecekan Tanggal & TTL (Tabel Pelamar)
if 'pelamar' in transformed_dfs:
    print("\n[PELAMAR] Pengecekan Parsing TTL & Tanggal:")
    display(transformed_dfs['pelamar'][['nama_lengkap', 'tempat_lahir', 'tanggal_lahir']].dropna(subset=['tanggal_lahir']).head(5))

# 2. Pengecekan Konversi Gaji (Tabel Pelamar)
if 'pelamar' in transformed_dfs:
    print("\n[PELAMAR] Pengecekan Konversi Gaji (String IDR -> BigInt):")
    display(transformed_dfs['pelamar'][['nama_lengkap', 'ekspektasi_gaji']].head(5))

# 3. Pengecekan Enum Status & Laptop (Tabel Pelamar)
if 'pelamar' in transformed_dfs:
    print("\n[PELAMAR] Pengecekan Mapping Enum (Status Nikah & Laptop):")
    display(transformed_dfs['pelamar'][['nama_lengkap', 'status_pernikahan', 'penggunaan_laptop']].head(5))

# 4. Pengecekan Tahun & IPK (Tabel Pelamar Sekolah)
if 'pelamar_sekolah' in transformed_dfs:
    print("\n[PELAMAR_SEKOLAH] Pengecekan Tahun Lulus (Latest) & IPK (Decimal):")
    display(transformed_dfs['pelamar_sekolah'][['nama_sekolah', 'tahun_lulus', 'ipk']].head(5))

# 5. Pengecekan Tanggal Kursus (Tabel Pelamar Kursus)
if 'pelamar_kursus' in transformed_dfs:
    print("\n[PELAMAR_KURSUS] Pengecekan Konversi Tanggal Kursus:")
    display(transformed_dfs['pelamar_kursus'][['nama_kursus', 'tanggal']].head(5))

# 6. Pengecekan Status Progress (Tabel Progres Pelamar)
if 'progres_pelamar' in transformed_dfs:
    print("\n[PROGRES_PELAMAR] Pengecekan Normalisasi Status (baru -> Baru):")
    display(transformed_dfs['progres_pelamar'][['status_progres_pelamar']].drop_duplicates())


🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)

[PELAMAR] Pengecekan Parsing TTL & Tanggal:


,nama_lengkap,tempat_lahir,tanggal_lahir
0,-,-,1970-01-01
1,Hartika Prawidaningrum Harahap,Sidoarjo,1970-01-01
2,sdasd,asd,1970-01-01
3,Ni Putu Jayanti Nirmala Pradnya Santosa,Surabaya,1970-01-01
4,Ni Putu Jayanti Nirmala Pradnya Santosa,Surabaya,1970-01-01



[PELAMAR] Pengecekan Konversi Gaji (String IDR -> BigInt):


,nama_lengkap,ekspektasi_gaji
0,-,0.0
1,Hartika Prawidaningrum Harahap,4550000.0
2,sdasd,0.0
3,Ni Putu Jayanti Nirmala Pradnya Santosa,0.0
4,Ni Putu Jayanti Nirmala Pradnya Santosa,0.0



[PELAMAR] Pengecekan Mapping Enum (Status Nikah & Laptop):


...

,nama_sekolah,tahun_lulus,ipk
0,SDN Ranggeh,2000,0.0
1,UINSA Surabaya,2000,0.0
2,UNIVERSITAS NEGERI SURABAYA,2000,0.0
3,UNIVERSITAS NEGERI SEBELAS MARET SURAKARTA,2000,0.0
4,SMAK Kolese Santo Yusup Malang,2000,0.0



[PELAMAR_KURSUS] Pengecekan Konversi Tanggal Kursus:


,nama_kursus,tanggal
0,Data Science,1970-01-01
1,Teachers development,1970-01-01
2,MAHIR MICROSOFT EXCEL DAN GOOGLE SHEET,1970-01-01
3,Online IELTS Writing Premium Batch 61,1970-01-01
4,Diklat Samisanov 70,1970-01-01



[PROGRES_PELAMAR] Pengecekan Normalisasi Status (baru -> Baru):


,status_progres_pelamar
0,Interview
3,Tahap Test
8,Baru
15,Diterima
71,Ditolak


In [7]:
# 3.1.3 Detail Perbandingan Kolom & Tipe Data (Side-by-Side)
print("\n🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE")
for old_t, new_t in hanif_tables_map:
    print(f"\n{'='*15} {old_t.upper()} ➔ {new_t.upper()} {'='*15}")
    
    df_old = pd.DataFrame(raw_data.get(old_t, []))
    df_new = transformed_dfs.get(new_t, pd.DataFrame())
    
    if not df_new.empty:
        comparison = []
        # table_mapping defined for each table
        table_mapping = {}
        if old_t == 'pengajuan': table_mapping = {'idpengajuan': 'id_pengajuan', 'idusers': 'id_user', 'keterangan': 'posisi', 'jumlah': 'jumlah', 'syarat': 'syarat', 'pertanyaan': 'pertanyaan', 'alur': 'alur_seleksi', 'test': 'daftar_tes', 'status': 'status', 'created_at': 'created_at'}
        elif old_t == 'histori_pengajuan': table_mapping = {'idhistori': 'id_verifikasi', 'idpengajuan': 'id_pengajuan', 'status': 'status_verifikasi_pengajuan', 'catatan': 'catatan', 'created_at': 'created_at'}
        elif old_t == 'pelamar': table_mapping = {'idpelamar': 'id_pelamar', 'idpengajuan': 'id_pengajuan', 'email': 'email_pelamar', 'nama': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jk': 'jenis_kelamin', 'ttl': 'tanggal_lahir', 'alamat': 'alamat_ktp', 'domisili': 'alamat_domisili', 'wa': 'nomor_wa', 'linkedin': 'akun_linkedin', 'ig': 'akun_instagram', 'fb': 'akun_facebook', 'sosmed': 'sosmed_lain', 'laptop': 'spesifikasi_laptop', 'internet': 'internet', 'kegiatan': 'kegiatan_sekarang', 'rencana': 'rencana_karir', 'mobilitas': 'mobilitas', 'info': 'sumber_info', 'wfo': 'siap_wfo', 'bergabung': 'tanggal_bergabung', 'jenis': 'kategori_pelamar', 'work': 'riwayat_kerja', 'ppdk': 'riwayat_pendidikan', 'pengalaman': 'pengalaman_bidang', 'wawasan': 'wawasan', 'sehat': 'riwayat_kesehatan', 'statusnikah': 'status_pernikahan', 'ajar': 'kemampuan_ajar', 'app': 'penguasaan_aplikasi', 'apps': 'aplikasi_lainnya', 'gunalaptop': 'penggunaan_laptop', 'toefl': 'skor_toefl', 'gaji': 'ekspektasi_gaji', 'link': 'tautan_berkas', 'resign': 'alasan_resign', 'hasiliq': 'skor_iq', 'piciq': 'foto_iq', 'picminat': 'foto_minat', 'picpribadi': 'foto_kepribadian', 'created_at': 'created_at'}
        elif old_t == 'pekerjaan': table_mapping = {'idpekerjaan': 'id_pelamar_kerja', 'idusers': 'id_pelamar', 'namaperusahaan': 'nama_perusahaan', 'periode': 'periode', 'jabatan': 'jabatan', 'jobdesk': 'deskripsi_kerja'}
        elif old_t == 'pendidikan': table_mapping = {'idpendidikan': 'id_pelamar_sekolah', 'idusers': 'id_pelamar', 'sekolah': 'nama_sekolah', 'jenjang': 'jenjang', 'prodi': 'prodi', 'tahun': 'tahun_lulus', 'ipk': 'ipk', 'organisasi': 'organisasi'}
        elif old_t == 'kursus': table_mapping = {'idkursus': 'id_pelamar_kursus', 'idusers': 'id_pelamar', 'nama': 'nama_kursus', 'tanggal': 'tanggal', 'deskripsi': 'deskripsi', 'lokasi': 'lokasi', 'nosertifikat': 'nomor_sertifikat'}
        elif old_t == 'pelamar_note': table_mapping = {'idnote': 'id_progres_pelamar', 'idpelamar': 'id_pelamar', 'idusers': 'id_user', 'status': 'status_progres_pelamar', 'note': 'catatan', 'link': 'tautan_file', 'pertanyaan': 'pertanyaan', 'created_at': 'created_at'}
        elif old_t == 'pelamar_users': table_mapping = {'idassign': 'id_rekrutmen', 'idpelamar': 'id_pelamar', 'idusers': 'id_user'}

        for old_col, new_col in table_mapping.items():
            comparison.append({
                'Old Column': old_col,
                'Old Type': str(df_old[old_col].dtype) if old_col in df_old.columns else "N/A",
                '➔': '➔',
                'New Column': new_col,
                'New Type': str(df_new[new_col].dtype) if new_col in df_new.columns else "N/A"
            })
        
        # Cek kolom baru yang tidak ada di mapping
        for col in df_new.columns:
            if col not in table_mapping.values():
                comparison.append({
                    'Old Column': '(KOLOM BARU / CUSTOM)',
                    'Old Type': '-',
                    '➔': '➔',
                    'New Column': col,
                    'New Type': str(df_new[col].dtype)
                })
        
        display(pd.DataFrame(comparison))
        print(f"\n--- SAMPLE DATA NEW (2 Baris) ---")
        display(df_new.head(2))
    else:
        print(f"⚠️ Tabel {new_t} kosong.")


🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE

=============== PENGAJUAN ➔ PENGAJUAN_KARYAWAN ===============


,Old Column,Old Type,➔,New Column,New Type
0,idpengajuan,int64,➔,id_pengajuan,N/A
1,idusers,str,➔,id_user,str
2,keterangan,str,➔,posisi,str
3,jumlah,str,➔,jumlah,str
4,syarat,str,➔,syarat,str
5,pertanyaan,str,➔,pertanyaan,str
6,alur,str,➔,alur_seleksi,str
7,test,str,➔,daftar_tes,str
8,status,str,➔,status,str
9,created_at,datetime64[us],➔,created_at,str



--- SAMPLE DATA NEW (2 Baris) ---


,id_user,posisi,jumlah,syarat,pertanyaan,alur_seleksi,daftar_tes,status,created_at
0,U00016,Part-Time Offline English Teacher,2,"<h5 class=""t-20 mb3"" style=""box-sizing: border...",<p>Info Jam Kerja dan Gaji :</p>\r\n<p>&nbsp;<...,<p>1. Mengisi form dan test melalui link: http...,<p>Mempersiapkan bahan micro teaching (MT) onl...,Diterima,2023-06-16 17:02:47
1,U00014,Magang Sales & Marketing,1,<p>1. Background pendidikan apa saja</p>\r\n<p...,<p>1. Komitmen kapan bisa mulai dan lama magan...,<p>1. Seleksi administrasi</p>\r\n<p>2. Probin...,<p>1. Buatlah desain poster sederhana program ...,Diterima,2023-07-04 13:52:25



=============== HISTORI_PENGAJUAN ➔ HISTORI_PENGAJUAN ===============


...

,id_pelamar,id_user,status_progres_pelamar,catatan,tautan_file,pertanyaan,created_at
0,177,U00001,Interview,,https://drive.google.com/drive/folders/17AhJjH...,-,2023-05-29 16:56:05
1,177,U00012,Interview,<p>testing</p>,-,-,2023-05-29 16:57:22



=============== PELAMAR_USERS ➔ REKRUTMEN_PELAMAR ===============


,Old Column,Old Type,➔,New Column,New Type
0,idassign,int64,➔,id_rekrutmen,N/A
1,idpelamar,str,➔,id_pelamar,Int64
2,idusers,str,➔,id_user,str



--- SAMPLE DATA NEW (2 Baris) ---


,id_pelamar,id_user
0,5,U00014
1,2,U00014


## 4. Export ke Pickle

In [8]:
file_name = 'fase_3_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

total_records_new = sum(len(df) for df in transformed_dfs.values())
total_records_old = sum(len(records) for records in raw_data.values())

migration_result = {
    'fase': 'fase_3',
    'script': 'script_hanif',
    'fase_num': 3,
    'status': 'ready_for_insert',
    'old_records_total': total_records_old,
    'new_records_total': total_records_new,
    'diff': total_records_new - total_records_old,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_3",
  "script": "script_hanif",
  "fase_num": 3,
  "status": "ready_for_insert",
  "old_records_total": 1144,
  "new_records_total": 1383,
  "diff": 239,
  "pickle_file": "fase_3_hanif.pkl",
  "timestamp": "2026-07-09T14:44:45.631342"
}


In [9]:
# --- EXPORT KE CSV UNTUK VERIFIKASI ---
EXPORT_TO_CSV = True  # Ubah ke False jika tidak ingin menghasilkan file CSV

if EXPORT_TO_CSV:
    import os
    import pandas as pd
    target_dir = "../extract/cek_csv"
    os.makedirs(target_dir, exist_ok=True)
    for tbl_name, df_tbl in transformed_dfs.items():
        csv_path = os.path.join(target_dir, f"{tbl_name}.csv")
        df_to_save = df_tbl.copy()
        
        # Clean any float ID/FK columns that contain .0 to pure integers
        for col in df_to_save.columns:
            col_lower = col.lower()
            is_id_col = col_lower.startswith('id_') or col_lower.endswith('_id') or col_lower == 'id' or 'id_' in col_lower or '_id_' in col_lower
            if is_id_col:
                non_nulls = df_to_save[col].dropna()
                if not non_nulls.empty:
                    try:
                        pd.to_numeric(non_nulls, errors='raise')
                        df_to_save[col] = pd.to_numeric(df_to_save[col], errors='coerce').round().astype('Int64')
                    except (ValueError, TypeError):
                        pass
        
        # Fix: Convert any StringDtype to object for clean serialization
        for col in df_to_save.columns:
            if str(df_to_save[col].dtype) in ['string', 'string[python]']:
                df_to_save[col] = df_to_save[col].astype(object)
        df_to_save.to_csv(csv_path, index=False)
        print(f"💾 Tabel {tbl_name} diekspor ke {csv_path} ({len(df_tbl)} baris)")
else:
    print("ℹ️ Ekspor ke CSV dinonaktifkan.")

💾 Tabel mapping_pengajuan_karyawan diekspor ke ../extract/cek_csv\mapping_pengajuan_karyawan.csv (33 baris)
💾 Tabel pengajuan_karyawan diekspor ke ../extract/cek_csv\pengajuan_karyawan.csv (33 baris)


💾 Tabel histori_pengajuan diekspor ke ../extract/cek_csv\histori_pengajuan.csv (79 baris)
💾 Tabel mapping_pelamar diekspor ke ../extract/cek_csv\mapping_pelamar.csv (192 baris)
💾 Tabel pelamar diekspor ke ../extract/cek_csv\pelamar.csv (192 baris)
💾 Tabel pelamar_kerja diekspor ke ../extract/cek_csv\pelamar_kerja.csv (67 baris)


💾 Tabel pelamar_sekolah diekspor ke ../extract/cek_csv\pelamar_sekolah.csv (53 baris)
💾 Tabel pelamar_kursus diekspor ke ../extract/cek_csv\pelamar_kursus.csv (50 baris)
💾 Tabel progres_pelamar diekspor ke ../extract/cek_csv\progres_pelamar.csv (403 baris)
💾 Tabel rekrutmen_pelamar diekspor ke ../extract/cek_csv\rekrutmen_pelamar.csv (281 baris)
